# Exploratory Data Analysis

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style = 'whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# First load the harmonized dataset
# project/data/harmonized/county_panel_raw.csv for deepnote
dataset = pd.read_csv('project/data/harmonized/county_panel_raw.csv')
display(dataset)

Rows 0, 1, 2, and 4 (Illinois counties), and the bottom rows (Puerto Rico): almost all the AQI columns are NaN

The EPA doesn't have air quality monitoring stations in every single county in the US. Many rural or smaller counties do not collect AQI data. Furthermore, CDC PLACES data might be missing for US territories like Puerto Rico (as seen by the NaN values in mental_health_prevalence).

I run a missing data assessment:

In [ ]:
# What percentage of data is missing per column?
print(dataset.isnull().sum() / len(dataset) * 100)

EDA Strategy: Create two "versions" of the dataset.

1. A broad dataset that focuses on ACS and PLACES metrics (where data is mostly complete)
2. A filtered dataset where nan rows in Median AQI are dropped to study the environment vs. health intersection (e.g. row 3)

Did the merge introduce systemic errors or extra rows?

In [ ]:
dataset.groupby('year').count()

Are all 50 states represented? Did some drop out during the merge?

In [ ]:
print(sorted(list(dataset.state_name.unique())))
print(len(dataset.state_name.unique()))

Hawaii and Puerto Rico are included. For completeness of the data, let's restrict to only 50 US States:

In [ ]:
dataset = dataset[(dataset['state_name'] != 'Hawaii') & (dataset['state_name'] != 'Puerto Rico')]

In [ ]:
print(dataset.state_name.unique())

In [ ]:
# How many complete cases do we actually have across all three sources?
complete_cases = dataset.dropna(subset=['Median AQI', 'mental_health_prevalence']).shape[0]
print(f"Rows with complete AQI and Health data: {complete_cases}")

# Which years are covered and is the data is balanced across time?
print(dataset['year'].value_counts())

# Let's do a quick statistical breakdown of the columns we can see
print(dataset[['median_income', 'mental_health_prevalence', 'Median AQI']].describe())

In [ ]:
dataset_cols = dataset.columns
print(dataset_cols)

## Feature Engineering

Raw counts (poverty_count, unemployed_count) are biased by county size, so let's calculate rates for these columns

In [ ]:
dataset['poverty_rate'] = dataset['poverty_count'] / dataset['population']
dataset['unemployed_rate'] = dataset['unemployed_count'] / dataset['population']
dataset['bad_air_day_ratio'] = (dataset['Unhealthy Days'] + dataset['Very Unhealthy Days'] + dataset['Hazardous Days']) / dataset['Days with AQI']
dataset['good_air_day_ratio'] = (dataset['Good Days'] + dataset['Moderate Days']) / dataset['Days with AQI']

In [ ]:
display(dataset)

In [ ]:
display(dataset.dropna(subset=['Median AQI', 'mental_health_prevalence']))

Do population and total_population match? First have to coerce to int type

In [ ]:
print(dataset[['population', 'total_population']].dtypes)

In [ ]:
dataset[['total_population']].tail()

In [ ]:
dataset[['total_population']] = (
    dataset[['total_population']]
    .replace(',', '', regex = True)
    .apply(pd.to_numeric, errors = 'coerce')
)

In [ ]:
dataset[['total_population']].tail()

In [ ]:
dataset[['population']].tail()

Let's just use the population column for this project.

## Univariate Analysis

### 1. Environmental (AQI)

- What is the typical air profile of an American county? What is the distribution of Median AQI and Max AQI? 
- Which pollutant is the dominant driver of bad air days across the country?

In [ ]:
# What is the typical air profile of an American county? What is the distribution of Median AQI and Max AQI? 
fig, axes = plt.subplots(1, 2, figsize = (20, 5))
fig.suptitle("Median and Max AQI Distribution Comparison")
axes[0].set_title("Median AQI Distribution")
axes[1].set_title("Max AQI Distribution")
sns.histplot(ax = axes[0], data = dataset, x = 'Median AQI', bins = 20, kde = True)
sns.histplot(ax = axes[1], data = dataset, x = 'Max AQI', bins = 20, kde = True)

plt.show()

In [ ]:
# Which pollutant is the dominant driver of bad air days across the country?
pollutants = ['Days CO', 'Days NO2', 'Days Ozone', 'Days PM2.5', 'Days PM10']

pollutant_totals = dataset[pollutants].sum()

sns.barplot(
    x = pollutant_totals.index,
    y = pollutant_totals.values
)

plt.xlabel('Pollutant')
plt.ylabel('Count')
plt.show()

### 2. Socioeconomic (ACS)

- Is median income normally distributed or heavily skewed to the left/right? 

In [ ]:
plt.figure(figsize = (15, 5))
sns.histplot(data = dataset, x = 'median_income', bins = 20, kde = True)
plt.show()

Heavily skewed to the right it seems? So a few incredibly wealthy counties?

### 3. Health (PLACES)

- What is the range of mental_health_prevalence?
- Is the variance between counties tight or wide? 

In [ ]:
# Descriptive Statistics
health_stats = dataset['mental_health_prevalence'].describe()
variance = dataset['mental_health_prevalence'].var()
iqr = health_stats['75%'] - health_stats['25%']

print(f"Minimum Prevalence: {health_stats['min']}%")
print(f"Maximum Prevalence: {health_stats['max']}%")
print(f"Absolute Range: {health_stats['max'] - health_stats['min']:.2f}%")
print(f"Average: {health_stats['mean']:.2f}%")
print(f"Median: {health_stats['50%']}%")
print(f"Variance: {variance:.2f}")
print(f"Interquartile Range: {iqr:.2f}% (middle 50% of counties)")

In [ ]:
# Let's visualize the spread
fig, axes = plt.subplots(1, 2)

sns.histplot(
    data = dataset, x = 'mental_health_prevalence',
    kde = True, bins = 30, ax = axes[0]
)
axes[0].axvline(health_stats['mean'], color = 'red', linestyle = '--', linewidth = 2, label = f"Mean ({health_stats['mean']:.1f}%)")
axes[0].axvline(health_stats['50%'], color = 'green', linestyle = '-', linewidth = 2, label = f"Median ({health_stats['50%']:.1f}%)")
axes[0].set_title("Distribution of Mental Health Prevalence Across US Counties")
axes[0].set_xlabel("Mental Health Prevalence (%)")
axes[0].set_ylabel("Count of Counties")
axes[0].legend()

sns.boxplot(
    data = dataset, x = 'mental_health_prevalence',
    ax = axes[1]
)
axes[1].set_title("Boxplot of Mental Health Prevalence")
axes[1].set_xlabel("Mental Health Prevalence (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Which counties are on extreme ends (outliers)? 
low_outliers = dataset.sort_values(by = 'mental_health_prevalence').head(3)
high_outliers = dataset.sort_values(by = 'mental_health_prevalence', ascending = False).head(3)

print(f"Three Counties with the Lowest Mental Health Prevalence: {low_outliers[['county_name', 'state_name', 'mental_health_prevalence']].to_string(index = False)}")
print(f"Three Counties with the Highst Mental Health Prevalence: {high_outliers[['county_name', 'state_name', 'mental_health_prevalence']].to_string(index = False)}")

Notes: 

- Mean and median lines align pretty much perfectly so the health data is normally distributed
- IQR is small. Most of US deals with a very similar baseline of mental health distress

## Bi & Multivariate Analysis

### Intersection 1: Environment vs. Health

Does bad air correlate with mental health? 

In [ ]:
df_int1 = dataset.dropna(subset = ['Median AQI', 'mental_health_prevalence']).copy()

In [ ]:
sns.regplot(
    data = df_int1, x = 'Median AQI', y = 'mental_health_prevalence'
)
plt.title("Air Quality vs. Mental Health Prevalence")
plt.xlabel("Median AQI")
plt.ylabel("Mental Health Prevalence (%)")
plt.show()

### Intersection 2: Environmental Justice

Do lower-income counties have worse air? 

In [ ]:
sns.regplot(
    data = df_int1, x = 'median_income', y = 'Median AQI'
)
plt.title("County Median Income vs. Median AQI")
plt.xlabel("Median Income (USD)")
plt.ylabel("Median AQI")
plt.show()

### Intersection 3: Income vs. AQI vs. Mental Health

In [ ]:
scatter = plt.scatter(
    data = df_int1, x = 'Median AQI', y = 'mental_health_prevalence', c = 'median_income',
    alpha = 0.6
)
cbar = plt.colorbar(scatter)
cbar.set_label("Median Income (USD)", rotation = 270, labelpad = 15)
plt.title("Interaction of Air Quality, Income, and Mental Health")
plt.xlabel("Median AQI")
plt.ylabel("Mental Health Prevalence (%)")
plt.show()

## Temporal Analysis 

Let's look at macro shifts over time (year)

In [ ]:
yearly_trends = df_int1.groupby("year")[['Median AQI', 'mental_health_prevalence']].mean().reset_index()

fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Year')
ax1.set_ylabel('Average Median AQI')
sns.lineplot(data = yearly_trends, x = 'year', y = 'Median AQI', color = color, ax = ax1, marker = 'o')
ax1.tick_params(axis = 'y', labelcolor = color)

color = 'tab:blue'
ax2 = ax1.twinx()
ax2.set_ylabel('Average Mental Health Prevalence (%)')
sns.lineplot(data = yearly_trends, x = 'year', y = 'mental_health_prevalence', color = color, ax = ax2, marker = 's')
ax2.tick_params(axis = 'y', labelcolor = color)

plt.title("Nationwide Temporal Trends: AQI vs. Mental Health")
fig.tight_layout()
plt.show()

## Spatial Analysis

Let's look at state-level aggregations

In [ ]:
state_geo = df_int1.groupby('state_name')[['mental_health_prevalence', 'Median AQI', 'poverty_rate']].mean().reset_index()
top_states_mental = state_geo.sort_values(by = 'mental_health_prevalence', ascending = False).head(10)

sns.barplot(data = top_states_mental, x = 'state_name', y = 'mental_health_prevalence')
plt.title("Top 10 States with Highest Average Mental Health Prevalence")
plt.xlabel("State")
plt.ylabel("Mental Health Prevalence (%)")
plt.xticks(rotation = 45)
plt.show()